In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Exact path of your dataset
file_path = '/content/drive/MyDrive/CODE_ALPHA_INTERNSHIP/Unemployment in India.csv'

# Read CSV file
df = pd.read_csv(file_path)

# Display first five rows
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# Remove unnecessary spaces from column names
df.columns = df.columns.str.strip()

# Remove extra spaces from text columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Replace empty strings with NaN
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# Check missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Convert numerical columns
numeric_cols = [
    'Estimated Unemployment Rate (%)',
    'Estimated Employed',
    'Estimated Labour Participation Rate (%)'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert Date column
df['Date'] = pd.to_datetime(
    df['Date'],
    dayfirst=True,
    errors='coerce'
)

# Remove rows with missing essential values
df.dropna(subset=[
    'Region',
    'Date',
    'Estimated Unemployment Rate (%)',
    'Estimated Employed',
    'Estimated Labour Participation Rate (%)',
    'Area'
], inplace=True)

# Remove duplicate records
df.drop_duplicates(inplace=True)

# Sort dataset by date
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)

print("\nDataset shape after cleaning:", df.shape)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

display(df.head())

In [ ]:
#Basic statistical analysis
unemployment_col = 'Estimated Unemployment Rate (%)'

print("Overall Average Unemployment Rate:",
      round(df[unemployment_col].mean(), 2), "%")

print("Maximum Unemployment Rate:",
      round(df[unemployment_col].max(), 2), "%")

print("Minimum Unemployment Rate:",
      round(df[unemployment_col].min(), 2), "%")

print("Standard Deviation:",
      round(df[unemployment_col].std(), 2))

print("\nStatistical Summary:")
display(df[numeric_cols].describe())

In [ ]:
#Overall unemployment distribution
plt.figure(figsize=(10, 6))

sns.histplot(
    df[unemployment_col],
    bins=30,
    kde=True
)

plt.title('Distribution of Unemployment Rate in India')
plt.xlabel('Unemployment Rate (%)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
#Overall unemployment trend
monthly_unemployment = (
    df.groupby('Date')[unemployment_col]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))

plt.plot(
    monthly_unemployment['Date'],
    monthly_unemployment[unemployment_col],
    marker='o',
    linewidth=2
)

plt.title('Monthly Unemployment Rate in India')
plt.xlabel('Date')
plt.ylabel('Unemployment Rate (%)')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(monthly_unemployment)

In [ ]:
#State-wise unemployment analysis
state_unemployment = (
    df.groupby('Region')[unemployment_col]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 8))

state_unemployment.plot(
    kind='bar',
    color='steelblue'
)

plt.title('Average Unemployment Rate by Region')
plt.xlabel('Region')
plt.ylabel('Average Unemployment Rate (%)')

plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

print("Top 10 regions with highest average unemployment:")
display(state_unemployment.head(10))

print("Top 10 regions with lowest average unemployment:")
display(state_unemployment.tail(10))

In [ ]:
#Rural vs Urban unemployment comparison
area_unemployment = (
    df.groupby('Area')[unemployment_col]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=area_unemployment,
    x='Area',
    y=unemployment_col
)

plt.title('Rural vs Urban Unemployment Rate')
plt.xlabel('Area')
plt.ylabel('Average Unemployment Rate (%)')

plt.tight_layout()
plt.show()

display(area_unemployment)

In [ ]:
#COVID-19 impact analysis
# Add year and month
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# Define periods
df['Period'] = np.where(
    df['Date'] < '2020-03-01',
    'Pre-COVID',
    np.where(
        df['Date'] >= '2020-04-01',
        'COVID Period',
        'Transition'
    )
)

# Compare pre-COVID and COVID periods
covid_analysis = (
    df[df['Period'].isin(['Pre-COVID', 'COVID Period'])]
    .groupby('Period')[unemployment_col]
    .mean()
    .reindex(['Pre-COVID', 'COVID Period'])
)

print("COVID-19 Impact Analysis:")
display(covid_analysis)

# Calculate difference
difference = (
    covid_analysis['COVID Period'] -
    covid_analysis['Pre-COVID']
)

print("Change in unemployment rate:",
      round(difference, 2), "percentage points")

# Visualization
plt.figure(figsize=(8, 5))

covid_analysis.plot(
    kind='bar',
    color=['skyblue', 'salmon']
)

plt.title('Impact of COVID-19 on Unemployment Rate')
plt.xlabel('Period')
plt.ylabel('Average Unemployment Rate (%)')

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
#Monthly COVID-19 impact trend
covid_monthly = (
    df.groupby('Date')[unemployment_col]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))

plt.plot(
    covid_monthly['Date'],
    covid_monthly[unemployment_col],
    marker='o',
    color='darkblue'
)

# Mark the lockdown period
plt.axvspan(
    pd.Timestamp('2020-03-01'),
    pd.Timestamp('2020-06-30'),
    color='red',
    alpha=0.15,
    label='COVID-19 period'
)

plt.title('Monthly Unemployment Trend During COVID-19')
plt.xlabel('Date')
plt.ylabel('Unemployment Rate (%)')

plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Seasonal unemployment pattern
df['Month_Name'] = df['Date'].dt.month_name()

month_order = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

seasonal_analysis = (
    df.groupby('Month_Name')[unemployment_col]
    .mean()
    .reindex(month_order)
    .dropna()
)

plt.figure(figsize=(12, 6))

seasonal_analysis.plot(
    kind='bar',
    color='teal'
)

plt.title('Monthly Pattern of Unemployment Rate')
plt.xlabel('Month')
plt.ylabel('Average Unemployment Rate (%)')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(seasonal_analysis)

In [ ]:
#Correlation analysis
correlation = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)

plt.title('Correlation Between Economic Variables')
plt.tight_layout()
plt.show()

display(correlation)

In [ ]:
# Employment and labour participation analysis
#Monthly average employment
monthly_employment = (
    df.groupby('Date')['Estimated Employed']
    .mean()
)

plt.figure(figsize=(12, 6))

plt.plot(
    monthly_employment.index,
    monthly_employment.values,
    marker='o'
)

plt.title('Monthly Estimated Employment in India')
plt.xlabel('Date')
plt.ylabel('Estimated Employed')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Labour participation rate
monthly_labour = (
    df.groupby('Date')['Estimated Labour Participation Rate (%)']
    .mean()
)

plt.figure(figsize=(12, 6))

plt.plot(
    monthly_labour.index,
    monthly_labour.values,
    marker='o',
    color='green'
)

plt.title('Monthly Labour Participation Rate')
plt.xlabel('Date')
plt.ylabel('Labour Participation Rate (%)')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Identify highest unemployment month
highest_month = monthly_unemployment.loc[
    monthly_unemployment[unemployment_col].idxmax()
]

lowest_month = monthly_unemployment.loc[
    monthly_unemployment[unemployment_col].idxmin()
]

print("Highest Unemployment Month:")
print(highest_month)

print("\nLowest Unemployment Month:")
print(lowest_month)

In [ ]:
#Generate final analytical report
print("=" * 55)
print("       UNEMPLOYMENT ANALYSIS REPORT - INDIA")
print("=" * 55)

print("\n1. Total observations:", len(df))

print("\n2. Average unemployment rate:",
      round(df[unemployment_col].mean(), 2), "%")

print("\n3. Highest unemployment rate:",
      round(df[unemployment_col].max(), 2), "%")

print("\n4. Lowest unemployment rate:",
      round(df[unemployment_col].min(), 2), "%")

print("\n5. Average rural and urban unemployment:")
print(area_unemployment.to_string(index=False))

print("\n6. COVID-19 impact:")
print(covid_analysis)

print("\n7. Highest unemployment region:")
print(state_unemployment.index[0],
      "-", round(state_unemployment.iloc[0], 2), "%")

print("\n8. Lowest unemployment region:")
print(state_unemployment.index[-1],
      "-", round(state_unemployment.iloc[-1], 2), "%")

print("\n9. Highest unemployment month:")
print(highest_month['Date'].strftime('%B %Y'))

print("\n10. Lowest unemployment month:")
print(lowest_month['Date'].strftime('%B %Y'))

print("\n" + "=" * 55)
print("              ANALYSIS COMPLETED")
print("=" * 55)